# Model Performance Evaluation

This notebook loads available model artifacts and computes performance statistics for the research results section. It focuses on available models in the project and extracts metrics from saved checkpoints and test data where possible.

In [ ]:
import os
import pickle
import torch
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt

print('Libraries loaded successfully')

Libraries loaded successfully


In [ ]:
missing = []
for pkg in ['requests', 'pyotp', 'selenium']:
    try:
        __import__(pkg)
    except ImportError:
        missing.append(pkg)

if missing:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install'] + missing)
    for pkg in missing:
        __import__(pkg)

print('Installed missing packages:', missing)


In [ ]:
# Locate available trained models and saved artifacts
base_path = Path('data/trained_models')
print('Base path exists:', base_path.exists())
print('Files found:')
for item in sorted(base_path.glob('*')):
    print(' -', item.name)

lstm_path = base_path / 'lstm_best.pth'
if not lstm_path.exists():
    raise FileNotFoundError(f'Missing {lstm_path}')

checkpoint = torch.load(lstm_path, map_location='cpu')
print('\nLoaded checkpoint keys:', list(checkpoint.keys()))
print('Train losses count:', len(checkpoint.get('train_losses', [])))
print('Val losses count:', len(checkpoint.get('val_losses', [])))
if checkpoint.get('train_losses'):
    print('First train loss:', checkpoint['train_losses'][0])
    print('Final train loss:', checkpoint['train_losses'][-1])
if checkpoint.get('val_losses'):
    print('First val loss:', checkpoint['val_losses'][0])
    print('Final val loss:', checkpoint['val_losses'][-1])
    print('Best val loss:', min(checkpoint['val_losses']))

print('Model state dict params:', sum(p.numel() for p in checkpoint['model_state_dict'].values()))

Base path exists: True
Files found:
 - lstm_best.pth
 - lstm_best_config.pkl
 - lstm_best_scaler.pkl
 - lstm_intraday.pth
 - lstm_intraday_config.pkl
 - lstm_intraday_scaler.pkl
 - rl_checkpoints

Loaded checkpoint keys: ['model_state_dict', 'optimizer_state_dict', 'train_losses', 'val_losses']
Train losses count: 18
Val losses count: 18
First train loss: 0.007365581781759598
Final train loss: 0.0012741469858432353
First val loss: 0.00021578273757378547
Final val loss: 4.91009013785515e-05
Best val loss: 4.91009013785515e-05
Model state dict params: 1372289


In [ ]:
missing = []
try:
    import requests
except ImportError:
    missing.append('requests')
try:
    import pyotp
except ImportError:
    missing.append('pyotp')
try:
    import selenium
except ImportError:
    missing.append('selenium')

print('missing libs before install:', missing)

if missing:
    import sys, subprocess
    print('kernel python:', sys.executable)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install'] + missing)
    for pkg in missing:
        __import__(pkg)

from trading_execution import fetch_historical_data
from ai_agent.ai_decision_engine import AIDecisionEngine
from ai_agent.feature_engineering import FeatureEngineer


def build_prediction_dataset(df, feature_engineer, scaler, feature_cols, lookback_period):
    df_features = feature_engineer.prepare_features(df)
    valid_cols = [c for c in feature_cols if c in df_features.columns]
    if len(df_features) < lookback_period + 1:
        raise ValueError('Not enough data for sequence evaluation')

    scaled = scaler.transform(df_features[valid_cols])
    X = []
    y = []
    for i in range(lookback_period, len(scaled) - 1):
        X.append(scaled[i - lookback_period:i])
        y.append(df_features['close'].iloc[i + 1])
    return np.array(X), np.array(y)


def price_direction_label(price_change_pct, threshold=0.5):
    if price_change_pct > threshold:
        return 'BUY'
    if price_change_pct < -threshold:
        return 'SELL'
    return 'HOLD'


def evaluate_directional_accuracy(engine, df, is_intraday=False, lookback_period=None):
    if is_intraday:
        lookback_period = lookback_period or engine.intraday_feature_engineer.lookback_period
        feature_engineer = engine.intraday_feature_engineer
        scaler = engine.intraday_scaler
        feature_cols = engine.intraday_feature_cols
        model = engine.intraday_model
    else:
        lookback_period = lookback_period or engine.feature_engineer.lookback_period
        feature_engineer = engine.feature_engineer
        scaler = engine.scaler
        feature_cols = engine.feature_cols
        model = engine.model

    if model is None or scaler is None or feature_cols is None:
        raise ValueError('Required model or scaler not loaded for evaluation')

    X, y_actual = build_prediction_dataset(df, feature_engineer, scaler, feature_cols, lookback_period)
    y_pred = model.predict(X).flatten()

    directions_actual = []
    directions_pred = []
    for actual, pred, i in zip(y_actual, y_pred, range(lookback_period, len(df) - 1)):
        current_price = df['close'].iloc[i]
        actual_change = ((actual - current_price) / current_price) * 100
        pred_change = ((pred - current_price) / current_price) * 100
        directions_actual.append(price_direction_label(actual_change))
        directions_pred.append(price_direction_label(pred_change))

    metrics = {
        'accuracy': accuracy_score(directions_actual, directions_pred),
        'precision_macro': precision_score(directions_actual, directions_pred, average='macro', zero_division=0),
        'recall_macro': recall_score(directions_actual, directions_pred, average='macro', zero_division=0),
        'f1_macro': f1_score(directions_actual, directions_pred, average='macro', zero_division=0),
        'confusion_matrix': confusion_matrix(directions_actual, directions_pred, labels=['BUY','HOLD','SELL']),
        'classification_report': classification_report(directions_actual, directions_pred, labels=['BUY','HOLD','SELL'], zero_division=0)
    }
    return metrics

print('Evaluation helper functions defined successfully')

ModuleNotFoundError: No module named 'pyotp'

In [ ]:
# Evaluate daily model on available historical data
symbol = 'NSE_EQ|INE467B01029'  # TCS

print(f'Fetching historical data for {symbol}...')
df = fetch_historical_data(symbol, interval='day', days=300)
print('Data shape:', df.shape)
print('Columns:', df.columns.tolist())

engine = AIDecisionEngine(use_gpu=False)

if engine.model is not None:
    try:
        metrics_daily = evaluate_directional_accuracy(engine, df, is_intraday=False)
        print('Daily model directional metrics:')
        print(pd.Series({
            'accuracy': metrics_daily['accuracy'],
            'precision_macro': metrics_daily['precision_macro'],
            'recall_macro': metrics_daily['recall_macro'],
            'f1_macro': metrics_daily['f1_macro']
        }))
        print(metrics_daily['classification_report'])
    except Exception as e:
        print('Daily model evaluation failed:', e)
else:
    print('Daily model is not loaded; cannot evaluate')

if engine.intraday_model is not None:
    try:
        df_intraday = fetch_historical_data(symbol, interval='30minute', days=30)
        print('Intraday data shape:', df_intraday.shape)
        metrics_intraday = evaluate_directional_accuracy(engine, df_intraday, is_intraday=True)
        print('Intraday model directional metrics:')
        print(pd.Series({
            'accuracy': metrics_intraday['accuracy'],
            'precision_macro': metrics_intraday['precision_macro'],
            'recall_macro': metrics_intraday['recall_macro'],
            'f1_macro': metrics_intraday['f1_macro']
        }))
        print(metrics_intraday['classification_report'])
    except Exception as e:
        print('Intraday model evaluation failed:', e)
else:
    print('No intraday model available for evaluation')